## 📂 Writing Files (Escritura de Datos desde Volúmenes)

Hasta este momento ya hemos explorado las dos primeras formas de escribir datos en **Delta Lake**, tanto a partir de **fuentes externas** (Amazon S3, Google Cloud Storage y Azure Data Lake Storage) como de **fuentes internas**, representadas por los **Volúmenes de Databricks**.

Sin embargo, todavía nos quedan por conocer dos enfoques adicionales para persistir información en una Delta Table:

* 🔄 **CTAS + READFILES** (Enfoque híbrido)
* 🐍 **PySpark**

---

### 🚀 ¿Por qué surge el enfoque híbrido?

En el capítulo anterior vimos que **USING + OPTIONS** permite crear una tabla directamente a partir de un archivo.

No obstante, este enfoque presenta una limitación importante: **solo permite crear External Tables**, debido que requiere especificar la ubicación física (`LOCATION`) donde se encuentran los datos.

Para superar esta limitación, podemos combinar **CTAS** con **READFILE**, obteniendo un enfoque híbrido que nos permite crear **Managed Tables**, aprovechando la inferencia automática del esquema y la flexibilidad que ofrece `READFILES` durante la lectura del archivo.

En este capítulo exploraremos este enfoque y entenderemos cuándo resulta conveniente utilizarlo frente a las alternativas vistas anteriormente.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("13WritingFiles2").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

### 🚀 CREATE TABLE AS SELECT (CTAs) + READFILES

Como vimos anteriormente, el enfoque **USING + OPTIONS** permite leer archivos desde un Volumen especificando diferentes opciones según el formato del archivo. Sin embargo, utilizado por sí solo presenta una limitación: únicamente permite crear **External Tables**.

Para superar esta limitación, podemos combinar **CTAS (Create Table As Select)** con **READFILES**, formando un enfoque híbrido que nos permitirá crear **Managed Tables** y, por lo tanto, aprovechar todas las capacidades que ofrece Delta Lake, como las **transacciones ACID, el versionamiento, el Time Travel**, entre otras.

---

#### 🔄 ¿Cómo funciona este enfoque?

En este capítulo seguiremos una estrategia de dos pasos:

1. 📄 Crear una **Vista** que encapsule toda la lógica de lectura mediante **READFILES**.

2. 🗄️ Utilizar **CTAS** para crear una **Managed Table** a partir de la consulta realizada sobre esa Vista.

De esta forma, combinamos la flexibilidad que ofrece **READFILES** durante la lectura de archivos con la simplicidad de **CTAS** para persistir los datos en una Delta Table administrada por Databricks.


#### ========= CSV =============

In [0]:
### PASO 1). DEFINIR VISTA TEMPORAL

spark.sql("""
          
          CREATE OR REPLACE TEMPORARY VIEW temp_view_csv_cta_readfiles AS
          SELECT *
          FROM read_files(
              '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_*.csv',
              format => 'csv',
              header => true,
              inferSchema => true
          )

          """)
print("Vista Temporal creada correctamente")

### PASO 2). ENCAPSULAR VISTA TEMPORAL DENTRO DE CTAs

spark.sql(f"""
          
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.table_csv_cta_readfiles
        AS
        SELECT *
        FROM temp_view_csv_cta_readfiles
         
          """)

print("Tabla creada correctamente a partir de un CTAs")

### PASO 3). VERIFICAMOS DATOS DE CTAs
spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.table_csv_cta_readfiles")

#### ========= JSON =============

In [0]:
### PASO 1). DEFINIR VISTA TEMPORAL

spark.sql("""
          
          CREATE OR REPLACE TEMPORARY VIEW temp_view_json_cta_readfiles AS
          SELECT *
          FROM read_files(
              '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_oneline_*.json',
              format => 'json'
          )

          """)
print("Vista Temporal creada correctamente")

### PASO 2). ENCAPSULAR VISTA TEMPORAL DENTRO DE CTAs

spark.sql(f"""
          
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.table_json_cta_readfiles
        AS
        SELECT *
        FROM temp_view_json_cta_readfiles
         
          """)

print("Tabla creada correctamente a partir de un CTAs")

### PASO 3). VERIFICAMOS DATOS DE CTAs
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.table_json_cta_readfiles"))

### 🚀🐍 PySpark

Hasta ahora hemos explorado diferentes formas de escribir información proveniente de **fuentes internas** para convertirla en **Delta Tables**.

Sin embargo, aún nos queda por conocer el último enfoque: **PySpark**.

PySpark nace de la combinación de **Python** y **Apache Spark**, ofreciendo una de las herramientas más utilizadas en Databricks para el procesamiento distribuido de grandes volúmenes de información.

Gracias a este enfoque, podemos leer, transformar y escribir datos utilizando la **API de DataFrames**, aprovechando toda la capacidad de procesamiento de Spark sin necesidad de trabajar exclusivamente con consultas SQL.

---

#### 🚀 ¿Cuándo utilizar PySpark?

PySpark resulta especialmente útil cuando:

* 🐍 Nos sentimos más cómodos trabajando con Python que con SQL.
* ⚙️ Necesitamos desarrollar transformaciones más complejas sobre los datos.
* 📊 Queremos aprovechar la API de DataFrames de Spark para procesar grandes volúmenes de información.
* 🚀 Buscamos combinar la simplicidad de Python con la potencia del motor distribuido de Apache Spark.

En esta sección veremos cómo utilizar **PySpark** para leer información desde una fuente de datos y escribirla posteriormente como una **Delta Table**, aprovechando todas las capacidades que ofrece Delta Lake.


#### ========= CSV =============

In [0]:
### PASO 1). VERIFICAR FUENTE DE ORIGEN DE DATOS

#### FUENTE DE DATOS INTERNA:
df_interna = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_*.csv")
# df_interna.show()

### PASO 2). CONVERTIR FUENTE DE DATOS A DELTA TABLE

df_interna.write.format("delta").mode("overwrite").saveAsTable("catalog_databricks_2026_de.schema_databricks_2026_de.table_csv_pyspark")
print("Tabla creada exitosamente")

### PASO 3). VERIFICAR INFORMACION
df = spark.read.table("catalog_databricks_2026_de.schema_databricks_2026_de.table_csv_pyspark")
df.show()

#### ========= JSON =============

In [0]:
### PASO 1). VERIFICAR FUENTE DE ORIGEN DE DATOS

#### FUENTE DE DATOS INTERNA:
df_interna = spark.read.format("json").load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_oneline_*.json")
# df_interna.show()

### PASO 2). CONVERTIR FUENTE DE DATOS A DELTA TABLE

df_interna.write.format("delta").mode("overwrite").saveAsTable("catalog_databricks_2026_de.schema_databricks_2026_de.table_json_pyspark")
print("Tabla creada exitosamente")

### PASO 3). VERIFICAR INFORMACION
df = spark.read.table("catalog_databricks_2026_de.schema_databricks_2026_de.table_json_pyspark")
df.show()